Understanding the problem

# 📌 Deep Learning with Stochastic Gradient Descent (SGD)

In [8]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np

# ✅ Load the MNIST Dataset

In [9]:
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])

train_dataset = torchvision.datasets.MNIST(root='./data', train=True, transform=transform, download=True)
test_dataset = torchvision.datasets.MNIST(root='./data', train=False, transform=transform, download=True)

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=1000, shuffle=False)

100%|██████████| 9.91M/9.91M [00:13<00:00, 720kB/s] 
100%|██████████| 28.9k/28.9k [00:00<00:00, 275kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 2.80MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 2.62MB/s]


# 🔹 Define a Fully Connected Neural Network 🔹

In [10]:
class NeuralNet(nn.Module):
    def __init__(self, input_size=28*28, hidden_size=128, output_size=10):
        super(NeuralNet, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_size, output_size)
    
    def forward(self, x):
        x = x.view(-1, 28*28)  # Flatten images
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x

# ✅ Model Initialization

In [11]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = NeuralNet().to(device)

# 🔹 Define Loss Function & Optimizers 🔹

In [12]:
criterion = nn.CrossEntropyLoss()

# Different SGD variants

In [13]:

optimizers = {
    "Vanilla SGD": optim.SGD(model.parameters(), lr=0.01),
    "SGD with Momentum": optim.SGD(model.parameters(), lr=0.01, momentum=0.9),
    "SGD with Nesterov": optim.SGD(model.parameters(), lr=0.01, momentum=0.9, nesterov=True),
    "Adam Optimizer": optim.Adam(model.parameters(), lr=0.01)
}

# 🔹 Training Function 🔹

In [14]:
def train_model(optimizer, num_epochs=5):
    model.train()
    loss_history = []

    for epoch in range(num_epochs):
        epoch_loss = 0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()
        
        avg_loss = epoch_loss / len(train_loader)
        loss_history.append(avg_loss)
        print(f"Epoch {epoch+1}/{num_epochs}, Loss: {avg_loss:.4f}")

    return loss_history

# ✅ Train & Compare Different Optimizers

In [15]:
loss_results = {}
for name, opt in optimizers.items():
    print(f"\nTraining with {name}...")
    model = NeuralNet().to(device) 
    optimizer = opt
    loss_results[name] = train_model(optimizer)


Training with Vanilla SGD...
Epoch 1/5, Loss: 2.3486
Epoch 2/5, Loss: 2.3486
Epoch 3/5, Loss: 2.3486
Epoch 4/5, Loss: 2.3486
Epoch 5/5, Loss: 2.3486

Training with SGD with Momentum...
Epoch 1/5, Loss: 2.3380
Epoch 2/5, Loss: 2.3379
Epoch 3/5, Loss: 2.3379
Epoch 4/5, Loss: 2.3379
Epoch 5/5, Loss: 2.3379

Training with SGD with Nesterov...
Epoch 1/5, Loss: 2.3318
Epoch 2/5, Loss: 2.3318
Epoch 3/5, Loss: 2.3318
Epoch 4/5, Loss: 2.3318
Epoch 5/5, Loss: 2.3318

Training with Adam Optimizer...
Epoch 1/5, Loss: 2.3285
Epoch 2/5, Loss: 2.3286
Epoch 3/5, Loss: 2.3286
Epoch 4/5, Loss: 2.3285
Epoch 5/5, Loss: 2.3286


# ✅ Evaluate the Model

In [16]:
def evaluate_model():
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    
    accuracy = 100 * correct / total
    print(f"Test Accuracy: {accuracy:.2f}%")
    return accuracy